# HuggingFace parquet rich-type fingerprinting

Goal: walk the parquet file-level KV metadata + Arrow schema, classify each column into a semantic type the viewer can render specially (image, audio, class label, translation, ...), and demo inline image rendering on top of that fingerprint.

Two-tier detection:
1. **Prefer** the `huggingface` KV key written by `datasets.Dataset.to_parquet()` and HF Hub's parquet exports.
2. **Fall back** to Arrow struct-shape heuristics for non-HF parquets.

In [1]:
"""Fingerprint walker. Two-tier: HF KV metadata, fall back to Arrow shape."""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any

import pyarrow as pa
import pyarrow.parquet as pq

HF_KV_KEY = "huggingface"

SEMANTIC_IMAGE = "image"
SEMANTIC_AUDIO = "audio"
SEMANTIC_VIDEO = "video"
SEMANTIC_PDF = "pdf"
SEMANTIC_NIFTI = "nifti"
SEMANTIC_CLASS_LABEL = "class_label"
SEMANTIC_TRANSLATION = "translation"
SEMANTIC_ARRAY_ND = "array_nd"
SEMANTIC_VALUE = "value"
SEMANTIC_LIST = "list"
SEMANTIC_UNKNOWN = "unknown"

HF_TYPE_TO_SEMANTIC = {
    "Image": SEMANTIC_IMAGE,
    "Audio": SEMANTIC_AUDIO,
    "Video": SEMANTIC_VIDEO,
    "Pdf": SEMANTIC_PDF,
    "Nifti": SEMANTIC_NIFTI,
    "ClassLabel": SEMANTIC_CLASS_LABEL,
    "Translation": SEMANTIC_TRANSLATION,
    "TranslationVariableLanguages": SEMANTIC_TRANSLATION,
    "Array2D": SEMANTIC_ARRAY_ND,
    "Array3D": SEMANTIC_ARRAY_ND,
    "Array4D": SEMANTIC_ARRAY_ND,
    "Array5D": SEMANTIC_ARRAY_ND,
    "Value": SEMANTIC_VALUE,
}


@dataclass
class FieldFingerprint:
    path: str
    semantic_type: str
    dtype: str | None = None
    hint: str | None = None
    names: list[str] | None = None
    list_depth: int = 0
    inner: "FieldFingerprint | None" = None
    extra: dict[str, Any] = field(default_factory=dict)


def _walk_hf_feature(name: str, feature: dict[str, Any]) -> FieldFingerprint:
    ftype = feature.get("_type")
    if ftype in {"List", "Sequence", "LargeList"}:
        sub = feature.get("feature", {})
        if isinstance(sub, dict) and "_type" in sub:
            inner = _walk_hf_feature(name, sub)
            return FieldFingerprint(
                path=name,
                semantic_type=SEMANTIC_LIST,
                list_depth=inner.list_depth + 1,
                inner=inner,
            )
        return FieldFingerprint(
            path=name,
            semantic_type=SEMANTIC_LIST,
            list_depth=1,
            extra={"sequence_of_struct": True},
        )

    semantic = HF_TYPE_TO_SEMANTIC.get(ftype, SEMANTIC_UNKNOWN)
    fp = FieldFingerprint(path=name, semantic_type=semantic)
    if ftype == "Value":
        fp.dtype = feature.get("dtype")
    elif ftype == "ClassLabel":
        fp.names = feature.get("names")
    elif ftype in {"Array2D", "Array3D", "Array4D", "Array5D"}:
        fp.extra = {"shape": feature.get("shape"), "dtype": feature.get("dtype")}
    elif ftype == "Audio":
        fp.extra = {"sampling_rate": feature.get("sampling_rate")}
    elif ftype in {"Translation", "TranslationVariableLanguages"}:
        fp.extra = {"languages": feature.get("languages")}
    elif semantic == SEMANTIC_UNKNOWN and ftype is not None:
        fp.extra = {"_type": ftype, "raw": feature}
    return fp


def _looks_like_image_struct(arrow_type: pa.DataType) -> bool:
    if not pa.types.is_struct(arrow_type):
        return False
    fields = {f.name: f.type for f in arrow_type}
    if set(fields) != {"bytes", "path"}:
        return False
    if not (pa.types.is_binary(fields["bytes"]) or pa.types.is_large_binary(fields["bytes"])):
        return False
    if not pa.types.is_string(fields["path"]):
        return False
    return True


def _walk_arrow_field(field_obj: pa.Field) -> FieldFingerprint:
    name = field_obj.name
    t = field_obj.type
    if pa.types.is_list(t) or pa.types.is_large_list(t):
        inner = _walk_arrow_field(pa.field(name, t.value_type))
        return FieldFingerprint(
            path=name,
            semantic_type=SEMANTIC_LIST,
            list_depth=inner.list_depth + 1,
            inner=inner,
        )
    if _looks_like_image_struct(t):
        return FieldFingerprint(path=name, semantic_type=SEMANTIC_IMAGE, hint="struct_shape")
    return FieldFingerprint(path=name, semantic_type=SEMANTIC_VALUE, dtype=str(t))


def fingerprint_parquet(path: str) -> dict[str, FieldFingerprint]:
    md = pq.read_metadata(path)
    kv = {k.decode(): v.decode() for k, v in (md.metadata or {}).items()}
    if HF_KV_KEY in kv:
        hf = json.loads(kv[HF_KV_KEY])
        features = hf.get("info", {}).get("features", {})
        return {name: _walk_hf_feature(name, feat) for name, feat in features.items()}
    schema = pq.read_schema(path)
    return {f.name: _walk_arrow_field(f) for f in schema}


def summary(fps: dict[str, FieldFingerprint]) -> str:
    lines = []
    for name, fp in fps.items():
        if fp.semantic_type == SEMANTIC_LIST and fp.inner:
            inner_t = fp.inner.semantic_type
            depth = f" (depth={fp.list_depth})" if fp.list_depth > 1 else ""
            lines.append(f"  {name}: list[{inner_t}]{depth}")
        elif fp.semantic_type == SEMANTIC_VALUE:
            lines.append(f"  {name}: {fp.dtype or 'value'}")
        elif fp.semantic_type == SEMANTIC_CLASS_LABEL:
            lines.append(f"  {name}: class_label ({len(fp.names or [])} classes)")
        elif fp.semantic_type == SEMANTIC_TRANSLATION:
            langs = (fp.extra or {}).get("languages") or "?"
            lines.append(f"  {name}: translation {langs}")
        elif fp.semantic_type == SEMANTIC_AUDIO:
            lines.append(f"  {name}: audio (sr={(fp.extra or {}).get('sampling_rate')})")
        else:
            lines.append(f"  {name}: {fp.semantic_type}")
    return "\n".join(lines)


print("fingerprint module ready")

fingerprint module ready


## Sample HF parquets and fingerprint each

Five datasets exercising different rich-type combinations. The fingerprint walker should detect: Image+ClassLabel (cifar10), ClassLabel+Value (glue/cola), Translation (opus-100), Audio+Value (librispeech), List[Image]+Value (MathNet).